# 왓챠피디아 평점 크롤링

Movie_Master의 영화 제목으로 왓챠피디아를 검색해 평점을 수집합니다.

- 제목 유사도 + 연도 기반 매칭
- `ratings_avg`, `ratings_count` 수집
- 출력: `data/02_interim/260506_watcha_ratings/watcha_ratings.csv`

In [18]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [19]:
# ★ _c_pdi 쿠키 값 만료되면 여기만 갱신
DEVICE_ID = 'web-vPnHq8ZT76Fuz7NZwVJIQrytV8YU6D'

HEADERS = {
    'accept'                      : 'application/vnd.frograms+json;version=2.1.0',
    'accept-language'             : 'ko-KR,ko;q=0.9',
    'user-agent'                  : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'x-frograms-app-code'         : 'Galaxy',
    'x-frograms-client'           : 'Galaxy-Web-App',
    'x-frograms-client-version'   : '2.1.0',
    'x-frograms-device-identifier': DEVICE_ID,
    'x-frograms-galaxy-language'  : 'ko',
    'x-frograms-version'          : '2.1.0',
    'referer'                     : 'https://pedia.watcha.com',
}

BASE     = Path('..').resolve()
DATA_IN  = BASE / 'data/02_interim/260504_promotion_split/promotion_0_movie_master_v2.csv'
DATA_OUT = BASE / 'data/02_interim/260506_watcha_ratings'
DATA_OUT.mkdir(exist_ok=True)

SLEEP_SEC  = 0.4
SIM_THRESH = 0.5

print('설정 완료')
print('입력 파일:', DATA_IN)
print('출력 폴더:', DATA_OUT)

설정 완료
입력 파일: C:\Users\USER\OneDrive\바탕 화면\AX git\ott-churn-prediction\kwon.donggeun\data\02_interim\260504_promotion_split\promotion_0_movie_master_v2.csv
출력 폴더: C:\Users\USER\OneDrive\바탕 화면\AX git\ott-churn-prediction\kwon.donggeun\data\02_interim\260506_watcha_ratings


In [20]:
def clean_title(t: str) -> str:
    t = re.sub(r'[\-\(\)\[\]【】（）]', ' ', str(t))
    return re.sub(r'\s+', ' ', t).strip()

def extract_year(t: str):
    m = re.search(r'\((\d{4})\)', str(t))
    return int(m.group(1)) if m else None

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_watcha(title: str, year=None):
    try:
        r = requests.get(
            'https://pedia.watcha.com/api/searches',
            params={'query': title, 'type': 'contents'},
            headers=HEADERS, timeout=10,
        )
        if r.status_code != 200:
            return None
        candidates = []
        result = r.json().get('result', {})
        for key in ['top_results', 'contents', 'movies']:
            candidates += result.get(key, [])
        best, best_score = None, 0.0
        for c in candidates:
            if c.get('content_type') != 'movies':
                continue
            sim = similarity(clean_title(title), clean_title(c.get('title', '')))
            yr_penalty = -abs(year - c['year']) * 0.05 if (year and c.get('year')) else 0
            score = sim + yr_penalty
            if score > best_score:
                best_score, best = score, c
        return best if best_score >= SIM_THRESH else None
    except Exception as e:
        print(f'  [검색 오류] {e}')
        return None

def get_rating(code: str):
    try:
        r = requests.get(
            f'https://pedia.watcha.com/api/contents/{code}',
            headers=HEADERS, timeout=10,
        )
        if r.status_code != 200:
            return None, None
        d = r.json().get('result', {})
        return d.get('ratings_avg'), d.get('ratings_count')
    except Exception as e:
        print(f'  [평점 오류] {e}')
        return None, None

print('함수 정의 완료')

함수 정의 완료


## 테스트 (5개)

In [21]:
df = pd.read_csv(DATA_IN, encoding='utf-8-sig')
print(f'총 영화 수: {len(df):,}개')
df.head()

총 영화 수: 3,874개


,MOVIE_NUM,movie_title,ott_release_month
0,3,그링고,202003
1,6,맨인블랙2,201205
2,9,손오공:색즉시공,201901
3,11,앵그리버드더무비,201606
4,12,백조공주:왕실결혼식,202008


In [22]:
# 5개 테스트
for _, row in df.head(5).iterrows():
    raw   = row['movie_title']
    year  = extract_year(raw)
    title = clean_title(raw)
    match = search_watcha(title, year)
    if match:
        avg, cnt = get_rating(match['code'])
        print(f'✅ {raw} → {match["title"]} ({match["year"]}) | 평점: {avg:.2f} ({cnt:,}명)' if avg else f'⚠️  {raw} → {match["title"]} | 평점 없음')
    else:
        print(f'❌ {raw} → 매칭 실패')
    time.sleep(0.5)

⚠️  그링고 → 그링고 | 평점 없음
⚠️  맨인블랙2 → 맨 인 블랙 2 | 평점 없음
⚠️  손오공:색즉시공 → 손오공: 색즉시공 | 평점 없음
⚠️  앵그리버드더무비 → 앵그리버드 더 무비 | 평점 없음
⚠️  백조공주:왕실결혼식 → 백조 공주 : 왕실 결혼식 | 평점 없음


## 전체 크롤링 (약 3시간)

In [23]:
results = []

for i, row in df.iterrows():
    raw   = row['movie_title']
    year  = extract_year(raw)
    title = clean_title(raw)
    match = search_watcha(title, year)

    if match:
        avg, cnt = get_rating(match['code'])
        results.append({
            'MOVIE_NUM'    : row['MOVIE_NUM'],
            '원제'         : raw,
            '왓챠_제목'    : match.get('title'),
            '왓챠_연도'    : match.get('year'),
            'watcha_code'  : match.get('code'),
            'ratings_avg'  : avg,
            'ratings_count': cnt,
        })
    else:
        results.append({
            'MOVIE_NUM'    : row['MOVIE_NUM'],
            '원제'         : raw,
            '왓챠_제목'    : None,
            '왓챠_연도'    : None,
            'watcha_code'  : None,
            'ratings_avg'  : None,
            'ratings_count': None,
        })

    if (i + 1) % 100 == 0:
        matched = sum(1 for r in results if r['ratings_avg'] is not None)
        print(f'[{i+1:,}/{len(df):,}] 매칭: {matched}개 ({matched/(i+1)*100:.1f}%)')

    time.sleep(SLEEP_SEC)

print('크롤링 완료!')

KeyboardInterrupt: 

In [ ]:
result_df = pd.DataFrame(results)
out_path  = DATA_OUT / 'watcha_ratings.csv'
result_df.to_csv(out_path, index=False, encoding='utf-8-sig')

matched = result_df['ratings_avg'].notna().sum()
print(f'매칭 성공: {matched:,}/{len(result_df):,}개 ({matched/len(result_df)*100:.1f}%)')
print(f'저장 완료: {out_path}')
result_df.head(10)